In [ ]:
import os
import json
import pandas as pd
import numpy as np
import geopandas as gpd
import subprocess
from tqdm import tqdm

In [ ]:
cur_dir = os.getcwd()

In [ ]:
pypsa_data_dir = os.path.join(cur_dir, "../../Data/eq_handoff_v5/handoff")
expansion_scenario_list = ["conservative", "bau", "aggressive"]
agg_type = "disagg"
eq_scenario_id = "bssc2014haywardhnhsshaw09mod_m7p32_se"
damage_data_dir = os.path.join(cur_dir, "../../Data/BSSC2014_scenarios")
eq_scenario_dir = os.path.join(damage_data_dir, eq_scenario_id)
pga_samples_all = pd.read_csv(os.path.join(eq_scenario_dir, "sites_gm_samples.csv"))

In [ ]:
R2D_E1_tempdir = os.path.join(cur_dir, "../../Data/SimCenterTemplates/tmp.SimCenter_E1")
R2D_E1_config_fp = os.path.join(R2D_E1_tempdir, "inputRWHALE.json")
with open(R2D_E1_config_fp, 'r') as f:
    R2D_E1_config = json.load(f)
pypsa_config = R2D_E1_config.copy()
num_samples = 32*32

In [ ]:
buses_list = []
# Buses / substations
for expansion_scenario in expansion_scenario_list:
    buses_all_fp = os.path.join(
        pypsa_data_dir,
        expansion_scenario,
        agg_type,
        "buses_all_aug.csv"
    )
    buses_df = pd.read_csv(buses_all_fp)
    buses_list.append(buses_df)
# Check if the buses in each scenario are the same
for i in range(1, len(buses_list)):
    common_buses = set(buses_list[0]['bus']).intersection(set(buses_list[i]['bus']))
    print(f"Common buses between scenario 0 and scenario {i}: {len(common_buses)}")
    if len(common_buses) != len(buses_list[0]['bus']):
        print(f"Scenario {i} has different buses than scenario 0")
        different_buses = set(buses_list[i]['bus']).difference(set(buses_list[0]['bus']))
        print(f"Buses in scenario {i} but not in scenario 0: {different_buses}")
        missing_buses = set(buses_list[0]['bus']).difference(set(buses_list[i]['bus']))
        print(f"Buses in scenario 0 but not in scenario {i}: {missing_buses}")

In [ ]:
# Create pelicun input for buses in each expansion scenario

# Create pelicun readable inventory csv files
buses_df['id'] = range(len(buses_df))
buses_df['id'] = buses_df['id'].astype(str)
# Pelicun assume the first column is 'id'
id_col = buses_df.pop('id')
buses_df.insert(0, 'id', id_col)

substation_df = buses_df.copy()
# Add "type" as "Substation" # one substation per bus
substation_df['type'] = "Substation"
# Add "Voltage" as v_nom
substation_df['Voltage'] = substation_df['v_nom_kv']
# Set "Anchored" as true
substation_df['Anchored'] = "true"
number_of_substations = len(substation_df)
common_dir = os.path.join(eq_scenario_dir, 'common')
if not os.path.exists(common_dir):
    os.makedirs(common_dir)
substation_info_csv_fn = "buses_info_pelicun.csv"
substation_df.to_csv(os.path.join(common_dir, 
    substation_info_csv_fn),
                      index=False)

In [ ]:
# Modify pypsa_config related to inventory files as needed
pypsa_config['Applications']['Assets']['Buildings'][
        'ApplicationData'
    ]['pathToSource'] = common_dir
pypsa_config['Applications']['Assets']['Buildings'][
    'ApplicationData'
    ]['assetSourceFile'] = substation_info_csv_fn
pypsa_config['Applications']['Assets']['Buildings'][
            'ApplicationData'
        ].pop('filter', None)
# Modify pypsa_config dl method
pypsa_config['Applications']['DL']['Buildings']['ApplicationData'][
        'DL_Method'
    ] = 'Hazus Earthquake - Electric Power'
# Modify pypsa_config variables to only save Damage
pypsa_config['SaveVariables'] = ['Demand','Damage']
# Modify number of damage state samples
pypsa_config['Applications']['DL']['Buildings'][
        'ApplicationData'
    ]['Realizations'] = num_samples

In [ ]:
# Create the gm sample file
pga_samples_substation = pd.DataFrame(
    # Manually make it two identical realization to avoid pelicun error
    # np.vstack([asset_gm_df['site_pga'].values.reshape(1, -1)]*2),
    pga_samples_all[substation_df['site_id'].astype(str).values.tolist()].values,
    columns=substation_df['id'].values
)
new_columns = pd.MultiIndex.from_product([pga_samples_substation.columns, ['PGA']])
pga_samples_substation.columns = new_columns
display(pga_samples_substation.head(3))

pga_samples_substation.to_csv(
    os.path.join(common_dir, "pga_samples_pelicun.csv"),
    index=False,
)
# Replace the RegionalMapping application name to UserDefinedDemandSample
pypsa_config['Applications']['RegionalMapping']['Buildings'][
        'Application'] = 'UserDefinedDemandSample'
pypsa_config['RegionalEvent']['eventFilePath'] = common_dir
pypsa_config['RegionalEvent']['eventFile'] = f'pga_samples_pelicun.csv'

pypsa_config_fp = os.path.join(
    common_dir, f"pypsa_config_pelicun.json"
)
with open(pypsa_config_fp, 'w') as f:
        json.dump(pypsa_config, f, indent=4)

In [ ]:
pga_samples_substation.max().max()

In [ ]:
# command to run pelicun
pelicun_path = os.path.join(cur_dir, "../../python_libs/pelicun_cli.py")
python_path = os.path.join(cur_dir, "../../.venv_pelicun/bin/python")

run_dir = common_dir
curdir = os.getcwd()
os.chdir(run_dir)
command = f"{python_path} -u {pelicun_path} regional_sim {pypsa_config_fp} -n 1"
try:
    subprocess.run(command, shell=True, check=True)
except subprocess.CalledProcessError as e:
    print(f"Command failed with error: {e}")

os.chdir(curdir)

In [ ]:
damage_id_to_bus_id = substation_df.set_index('id')['bus'].to_dict()

substation_damage_df = pd.read_csv(os.path.join(
    common_dir, "damage.csv"
))
substation_damage_df['bus'] = substation_damage_df['id'].map(damage_id_to_bus_id)

substation_damage_df.to_csv(os.path.join(common_dir, "damage_substation.csv"), index=False)
# Get remaining functional capacity according to Table 8-28 in HAZUS 6.1
substation_damage_to_remaining_functional_capacity = {
    0: 1.0,
    1: 0.5,
    2: 0.09,
    3: 0.04,
    4: 0.03
}
generator_damage_to_remaining_functional_capacity = {
    0: 1.0,
    1: 1.0,
    2: 0.24,
    3: 0.16,
    4: 0.02
}
storage_damage_to_remaining_functional_capacity = generator_damage_to_remaining_functional_capacity
for i in tqdm(range(num_samples), total=num_samples):
    col_name = str(i)
    substation_damage_df[col_name] = substation_damage_df[col_name].map(
        substation_damage_to_remaining_functional_capacity)
substation_damage_df.to_csv(os.path.join(common_dir, "capacity_substation.csv"), index=False)

Assets in individual expansion scenarios

In [ ]:
assets_df_list = []
columns_pelicun_needs = [
    'lon', 'lat', 'asset_type', 'v_nom_kv', 'p_nom_mw', 'asset_id', 'site_id', 'asset_key'
]

for expansion_scenario in expansion_scenario_list:
    assets_fp = os.path.join(
            pypsa_data_dir,
            expansion_scenario,
            agg_type,
            "assets_all_aug.csv"
        )
    assets_df = pd.read_csv(assets_fp)
    assets_df['asset_id'] = range(len(assets_df))
    asset_df_pelicun = assets_df[columns_pelicun_needs].copy()

    combined_assets = buses_df[['lon', 'lat', 'v_nom_kv', 'bus', 'site_id']].copy()
    for col in columns_pelicun_needs:
        if col not in combined_assets.columns:
            combined_assets[col] = None
    for col in combined_assets.columns:
        if col not in asset_df_pelicun.columns:
            asset_df_pelicun[col] = None

    combined_assets.loc[:, 'asset_type'] = 'Substation'
    combined_assets = pd.concat([combined_assets, asset_df_pelicun], ignore_index=True)
    combined_assets.loc[:, 'id'] = range(len(combined_assets))

    # Create pelicun readable inventory csv files
    combined_assets['id'] = combined_assets['id'].astype(str)
    # Pelicun assume the first column is 'id'
    id_col = combined_assets.pop('id')
    combined_assets.insert(0, 'id', id_col)

    mask_substation = combined_assets['asset_type'] == 'Substation'
    substation_df = combined_assets[mask_substation].copy()
    # Add "type" as "Substation" # one substation per bus
    substation_df.loc[mask_substation, 'type'] = "Substation"
    # Add "Voltage" as v_nom
    substation_df.loc[mask_substation, 'Voltage'] = substation_df['v_nom_kv']
    # Set "Anchored" as true
    substation_df.loc[mask_substation, 'Anchored'] = "true"
    number_of_substations = mask_substation.sum()

    # generation
    mask_generator = combined_assets['asset_type'] == 'generator'
    print("Number of generators with  zero p_nom_mw:", len(combined_assets[mask_generator & (combined_assets['p_nom_mw'] == 0)]))
    number_of_generators = mask_generator.sum()

    # Add "type" as "Generation"
    generation_df = combined_assets[mask_generator].copy()
    generation_df.loc[:, 'type'] = "Generation"
    # Add "Output" as v_nom
    generation_df.loc[:, 'Output'] = generation_df['p_nom_mw'].apply(lambda x: f"{x*1e6} w")
    # Set "Anchored" as true
    generation_df.loc[:, 'Anchored'] = "true"

    # storage unit
    mask_storage = combined_assets['asset_type'] == 'storage_unit'
    storage_df = combined_assets[mask_storage].copy()
    # Add "type" as "Generation" (Use the same fragility as small generation plants)
    storage_df.loc[mask_storage, 'type'] = "Generation"
    # Add "Capacity" as v_nom
    storage_df.loc[mask_storage, 'Output'] = storage_df['p_nom_mw'].apply(lambda x: f"{x*1e6} w")
    # Set "Anchored" as true
    storage_df.loc[mask_storage, 'Anchored'] = "true"

    combined_assets = pd.concat([substation_df, generation_df, storage_df], ignore_index=True)

    asset_info_csv_dir = os.path.join(eq_scenario_dir, expansion_scenario)
    if not os.path.exists(asset_info_csv_dir):
        os.makedirs(asset_info_csv_dir)
    asset_info_csv_fn = "asset_info_pelicun.csv"
    combined_assets.to_csv(os.path.join(asset_info_csv_dir, 
        asset_info_csv_fn),
                        index=False)
    assets_df_list.append(combined_assets)

In [ ]:
for es_i, expansion_scenario in enumerate(expansion_scenario_list):
    asset_info_csv_dir = os.path.join(eq_scenario_dir, expansion_scenario)
    asset_info_csv_fn = "asset_info_pelicun.csv"

    # Modify pypsa_config related to inventory files as needed
    pypsa_config['Applications']['Assets']['Buildings'][
            'ApplicationData'
        ]['pathToSource'] = asset_info_csv_dir
    pypsa_config['Applications']['Assets']['Buildings'][
        'ApplicationData'
        ]['assetSourceFile'] = asset_info_csv_fn
    pypsa_config['Applications']['Assets']['Buildings'][
                'ApplicationData'
            ].pop('filter', None)
    # Modify pypsa_config dl method
    pypsa_config['Applications']['DL']['Buildings']['ApplicationData'][
            'DL_Method'
        ] = 'Hazus Earthquake - Electric Power'
    # Modify pypsa_config variables to only save Damage
    pypsa_config['SaveVariables'] = ['Demand','Damage']
    # Modify number of damage state samples
    pypsa_config['Applications']['DL']['Buildings'][
            'ApplicationData'
        ]['Realizations'] = num_samples

    assets = assets_df_list[es_i]

    # Create the gm sample file
    pga_samples = pd.DataFrame(
        # Manually make it two identical realization to avoid pelicun error
        # np.vstack([asset_gm_df['site_pga'].values.reshape(1, -1)]*2),
        pga_samples_all[assets['site_id'].astype(str).values.tolist()].values,
        columns=assets['id'].values
    )
    new_columns = pd.MultiIndex.from_product([pga_samples.columns, ['PGA']])
    pga_samples.columns = new_columns

    pga_samples.to_csv(
        os.path.join(asset_info_csv_dir, "pga_samples.csv"),
        index=False,
    )
    # Replace the RegionalMapping application name to UserDefinedDemandSample
    pypsa_config['Applications']['RegionalMapping']['Buildings'][
            'Application'] = 'UserDefinedDemandSample'
    pypsa_config['RegionalEvent']['eventFilePath'] = asset_info_csv_dir
    pypsa_config['RegionalEvent']['eventFile'] = f'pga_samples.csv'

    pypsa_config_fp = os.path.join(
        asset_info_csv_dir, f"pypsa_config_pelicun.json"
    )
    with open(pypsa_config_fp, 'w') as f:
            json.dump(pypsa_config, f, indent=4)

In [ ]:
for es_i, expansion_scenario in enumerate(expansion_scenario_list):
    asset_info_csv_dir = os.path.join(eq_scenario_dir, expansion_scenario)
    pypsa_config_fp = os.path.join(asset_info_csv_dir, "pypsa_config_pelicun.json")

    # command to run pelicun
    pelicun_path = "/resnick/groups/enceladus/jyzhao/CAPowerResilience/python_libs/pelicun_cli.py"
    python_path = "/resnick/groups/enceladus/jyzhao/CAPowerResilience/.venv_pelicun/bin/python"

    run_dir = asset_info_csv_dir
    curdir = os.getcwd()
    os.chdir(run_dir)
    command = f"{python_path} -u {pelicun_path} regional_sim {pypsa_config_fp} -n 1"
    try:
        subprocess.run(command, shell=True, check=True)
    except subprocess.CalledProcessError as e:
        print(f"Command failed with error: {e}")

    os.chdir(curdir)

In [ ]:
for es_i, expansion_scenario in enumerate(expansion_scenario_list):
    asset_info_csv_dir = os.path.join(eq_scenario_dir, expansion_scenario)
    assets_df = assets_df_list[es_i]
    assets_df['id'] = assets_df['id'].astype(int)
    damage_id_to_asset_id = assets_df.set_index('id')['asset_id'].to_dict()
    damage_id_to_bus_id = assets_df.set_index('id')['bus'].to_dict()
    damage_id_to_asset_key = assets_df.set_index('id')['asset_key'].to_dict()

    damage_df_all = pd.read_csv(os.path.join(
        asset_info_csv_dir, "damage.csv"
    ))

    substation_damage_df = damage_df_all[damage_df_all['id'].isin(
        assets_df[assets_df['asset_type'] == 'Substation']['id']
    )].copy()
    substation_damage_df['bus'] = substation_damage_df['id'].map(damage_id_to_bus_id)

    generator_damage_df = damage_df_all[damage_df_all['id'].isin(
        assets_df[assets_df['asset_type'] == 'generator']['id']
    )].copy()
    generator_damage_df['asset_id'] = generator_damage_df['id'].map(damage_id_to_asset_id)
    generator_damage_df['asset_key'] = generator_damage_df['id'].map(damage_id_to_asset_key)

    storage_damage_df = damage_df_all[damage_df_all['id'].isin(
        assets_df[assets_df['asset_type'] == 'storage_unit']['id']
    )].copy()
    storage_damage_df['asset_id'] = storage_damage_df['id'].map(damage_id_to_asset_id)
    storage_damage_df['asset_key'] = storage_damage_df['id'].map(damage_id_to_asset_key)

    substation_damage_df.to_csv(os.path.join(asset_info_csv_dir, "damage_substation.csv"), index=False)
    generator_damage_df.to_csv(os.path.join(asset_info_csv_dir, "damage_generator.csv"), index=False)
    storage_damage_df.to_csv(os.path.join(asset_info_csv_dir, "damage_storage.csv"), index=False)

    for i in tqdm(range(num_samples), total=num_samples):
        col_name = str(i)
        substation_damage_df[col_name] = substation_damage_df[col_name].map(
            substation_damage_to_remaining_functional_capacity)
        generator_damage_df[col_name] = generator_damage_df[col_name].map(
            generator_damage_to_remaining_functional_capacity)
        storage_damage_df[col_name] = storage_damage_df[col_name].map(
            storage_damage_to_remaining_functional_capacity)

    substation_damage_df.to_csv(os.path.join(asset_info_csv_dir, "capacity_substation.csv"), index=False)
    generator_damage_df.to_csv(os.path.join(asset_info_csv_dir, "capacity_generator.csv"), index=False)
    storage_damage_df.to_csv(os.path.join(asset_info_csv_dir, "capacity_storage.csv"), index=False)